In [13]:
import os
import pandas as pd

from advisor_backend.interface import Interface
import numpy as np

# 算子调优分析
## 1. 算子分析的数据准备
当前算子分析工具支持分析Ascend Pyorch Profiler方式生成的ascend_pt目录
## 2. 融合算子分析
当前支持分析模型中存在可融合的小算子，并给出优化建议。

"更多融合算子信息，请查阅 https://www.hiascend.com/document/detail/zh/CANNCommunityEdition/700alpha003/processormodel/hardwaredesc_0001.html

## 3. 异常性能算子分析
支持分析模型中性能异常的计算算子

In [14]:
# EDIT THE PROFILING DATA PATH
compute_path = "[YOUR PATH]"
interface = Interface(compute_path)
data = interface.get_data('compute', 'npu_fused')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 900)
display(data['data'].iloc[:, :-2])
print('\n')
print(data['bottleneck'])
print('\n')
print(data['advice'])

[INFO] Start to analyse the target file: D:\work\ascend_pt\ASCEND_PROFILER_OUTPUT\kernel_details.csv


,pattern_name,pattern,len,count,duration sum(us),op durations(us),index
18,torch_npu.npu_swiglu,"(Slice, Slice, Swish, Mul)",4,1,27.53,"[21.2, 0.05, 3.14, 3.14]",[0]




The computing time of fusable op is 27.53 ms.


Advice 0:
Replace [Slice, Slice, Swish, Mul] with torch_npu.npu_swiglu. This pattern first happened in: 
/root/torch/module.py
/root/test/slice.py(116)


In [15]:
# 异常性能算子识别
from advisor_backend.compute_advice.npu_slow_advice import NpuSlowAdvice

npu_slow_advice = NpuSlowAdvice(compute_path)
data = interface.get_data('compute', 'npu_slow')
slow_op_data = data[data["color"] == "RED"]
display(slow_op_data)

[INFO] Start to analyse the target file: D:\work\ascend_pt\ASCEND_PROFILER_OUTPUT\kernel_details.csv


,Step Id,Model ID,Task ID,Stream ID,Name,Type,Accelerator Core,Start Time(us),Duration(us),Wait Time(us),Block Dim,Mix Block Dim,Input Shapes,Input Data Types,Input Formats,Output Shapes,Output Data Types,Output Formats,Context ID,aicore_time(us),aic_total_cycles,aic_mac_ratio,aic_mac_int8_ratio,aic_cube_fops,aic_vector_fops,aiv_time(us),aiv_total_cycles,aiv_vec_fp32_ratio,aiv_vec_fp16_ratio,aiv_vec_int32_ratio,aiv_vec_misc_ratio,aiv_cube_fops,aiv_vector_fops,size(MB),throughput(GB/s),color
0,1,4294967295,1265,16,Slice1,Slice,AI_VECTOR_CORE,1699529623106750,21.20,261.56,9,0,"4,1025",INT64,FORMAT_ND,"4,1025",INT32,FORMAT_ND,NaN,0.0,0.0,0.0,0.0,0.0,0.0,1.77,29508.0,0.0,0.0,0.0062,0.0,0.0,5856.0,0.046921,2.161371,RED
4,1,4294967295,1265,16,Add1,Add,AI_CORE,1699529623106754,3.14,261.56,9,0,"4,1025",INT64,FORMAT_ND,"4,1025",INT32,FORMAT_ND,NaN,2.3,28888.0,0.2,0.1,0.1,0.7,1.77,29508.0,0.0,0.0,0.0062,0.0,0.0,5856.0,0.046921,14.592698,RED


In [16]:
NpuSlowAdvice.save_to_excel(data, file_path=os.path.join(compute_path, "slow_op.xlsx"))

In [17]:
# 异常性能算子call stack
call_stack = npu_slow_advice.get_call_stack(data, index_id=0, ts_col="Start Time(us)")
print("call stack: ")
print(call_stack)

call stack: 
/root/torch/module.py
/root/test/slice.py(116)
